# Seminari 4 · Sistemet e prodhimit

**CIS525 · Inteligjenca Artificiale · UET**

Ky ushtrim përdor pesë pyetje me alternativa për të mbledhur shenjat e një problemi të makinës. Faktet, pyetjet dhe rregullat janë në shqip, në `../data/seminar4/`. Motori `ProductionSystem` nxjerr përfundime me inferencë përpara.

Zgjidh **Restart Kernel and Run All Cells**, pastaj plotëso formularin **Çfarë problemi ka makina juaj?** në pjesën 4. Run All përfundon pa pritur hyrje nga tastiera; pyetësori pret zgjedhjet e tua në notebook. Nise nga rrënja e projektit ose `notebooks/`.

Shembulli është mësimor, për makina me motor me djegie të brendshme. Përfundimet tregojnë mundësi për kontroll, jo diagnozë përfundimtare. Detyra e plotë e seminarit mbetet në punë.


In [1]:
import json
from itertools import product
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display
from IPython.utils.capture import capture_output

pd.set_option("display.max_colwidth", 100)


def find_data_dir(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for directory in (start, *start.parents):
        if (directory / "data").is_dir():
            return directory / "data"
    raise FileNotFoundError("Nuk u gjet dosja data/. Nise notebook-un nga projekti ose notebooks/.")


DATA_DIR = find_data_dir()


## 1. Modeli i inferencës

- **Memoria e punës:** faktet e mbledhura nga përgjigjet dhe përfundimet e nxjerra.
- **Rregulli:** `kushtet → perfundimet`. Të gjitha kushtet e një rregulli duhet të jenë të pranishme.
- **Inferenca përpara:** faktet e reja mund të plotësojnë rregulla të tjera, derisa të mos shtohet më asgjë.

Klasa e starter-it mbetet motori i ekzekutimit. Emrat e brendshëm `conditions` dhe `actions` përshtaten nga fushat shqip të skedarit JSON. Për çdo pyetësor krijohet sistem i ri, që një vlerësim të mos trashëgojë përgjigjet e mëparshme.


In [2]:
class ProductionSystem:
    def __init__(self):
        self.working_memory = set()  # Stores facts, acting as working memory
        self.rules = []  # Stores production rules

    def add_fact(self, fact):
        """Adds a fact to the working memory."""
        self.working_memory.add(fact)

    def add_rule(self, conditions, actions):
        """Adds a rule to the system.
        conditions: A set of facts that must be present in working_memory for the rule to fire.
        actions: A set of facts to add to working_memory when the rule fires.
        """
        self.rules.append({'conditions': conditions, 'actions': actions})

    def infer(self):
        """Executes the inference cycle until no more rules can be fired."""
        fired_a_rule = True
        while fired_a_rule:
            fired_a_rule = False
            for rule in self.rules:
                # Check if all conditions of the rule are in working_memory
                if rule['conditions'].issubset(self.working_memory):
                    # If conditions are met, apply the actions
                    new_facts_added = False
                    for action in rule['actions']:
                        if action not in self.working_memory:
                            self.add_fact(action)
                            new_facts_added = True
                    if new_facts_added:
                        fired_a_rule = True
                        # For simplicity, we re-evaluate all rules after a change
                        # In a more complex system, conflict resolution strategies would be used
                        break  # Restart rule evaluation after a rule fires

## 2. Të dhënat dhe baza e njohurive

- `facts.json`: fjalori i 13 fakteve të vëzhgueshme; këto **nuk shtohen automatikisht** si simptoma.
- `questions.json`: pesë pyetje me alternativa për ndezjen, dritat, karburantin, temperaturën dhe gomat.
- `rules.json`: 14 rregulla me identifikues, kushte, përfundime, lloj dhe shpjegim.

Çdo pyetje lejon vetëm një alternativë. **Nuk e di** nuk shton fakt dhe nuk nënkupton mungesë problemi. Përfundimet dhe këshillat janë të ndara nga vëzhgimet.

Për shembull, rrotullimi i ngadaltë dhe dritat e dobëta mbështesin dyshimin për baterinë ose lidhjet elektrike ([RAC: shenjat e baterisë së dobët](https://www.rac.co.uk/drive/advice/car-maintenance/signs-your-car-battery-is-low/)). Rregullat janë thjeshtime për të ilustruar inferencën, jo një bazë e plotë diagnostikimi.


In [3]:
def load_knowledge(facts_text, rules_text, questions_text):
    """Valido fjalorin e fakteve, rregullat dhe pesë pyetjet përpara përdorimit."""
    def string_set(values, location):
        if not isinstance(values, list) or not values:
            raise ValueError(f"{location}: kërkohet listë jo bosh.")
        if any(not isinstance(value, str) or not value.strip() for value in values):
            raise ValueError(f"{location}: kërkohen tekste jo bosh.")
        if len(set(values)) != len(values):
            raise ValueError(f"{location}: nuk lejohen përsëritje.")
        return set(values)

    facts = string_set(json.loads(facts_text), "facts.json")
    raw_rules, questionnaire = json.loads(rules_text), json.loads(questions_text)
    if not isinstance(raw_rules, list) or not raw_rules:
        raise ValueError("rules.json: kërkohet listë jo bosh rregullash.")
    parsed_rules, rule_ids, conclusions = [], set(), set()
    for record in raw_rules:
        fields = {"id", "kushtet", "perfundimet", "lloji", "shpjegimi"}
        if not isinstance(record, dict) or set(record) != fields:
            raise ValueError("rules.json: mungojnë ose teprojnë fusha të rregullit.")
        if not isinstance(record["id"], str) or not record["id"].strip() or record["id"] in rule_ids:
            raise ValueError("rules.json: identifikuesi i rregullit duhet të jetë unik dhe jo bosh.")
        if record["lloji"] not in ("ndermjetem", "problem", "keshille"):
            raise ValueError("rules.json: lloji i rregullit është i panjohur.")
        if not isinstance(record["shpjegimi"], str) or not record["shpjegimi"].strip():
            raise ValueError("rules.json: kërkohet shpjegim.")
        conditions = string_set(record["kushtet"], record["id"] + " kushtet")
        actions = string_set(record["perfundimet"], record["id"] + " perfundimet")
        parsed_rules.append(record | {"kushtet": conditions, "perfundimet": actions})
        rule_ids.add(record["id"])
        conclusions.update(actions)
    if facts & conclusions:
        raise ValueError("Faktet e vëzhguara duhet të ndahen nga përfundimet e nxjerra.")
    if any(not rule["kushtet"] <= facts | conclusions for rule in parsed_rules):
        raise ValueError("Një kusht nuk është fakt i njohur ose përfundim i ndonjë rregulli.")

    if not isinstance(questionnaire, list) or len(questionnaire) != 5:
        raise ValueError("questions.json: kërkohen saktësisht pesë pyetje.")
    question_ids, covered_facts = set(), set()
    for question in questionnaire:
        if not isinstance(question, dict) or set(question) != {"id", "pyetja", "alternativat"}:
            raise ValueError("questions.json: struktura e pyetjes është e pavlefshme.")
        if not isinstance(question["id"], str) or not question["id"].strip() or question["id"] in question_ids:
            raise ValueError("questions.json: identifikuesit duhet të jenë unikë dhe jo bosh.")
        if not isinstance(question["pyetja"], str) or not question["pyetja"].strip():
            raise ValueError("questions.json: teksti i pyetjes nuk mund të jetë bosh.")
        options = question["alternativat"]
        if not isinstance(options, list) or len(options) < 2:
            raise ValueError("questions.json: çdo pyetje kërkon alternativa.")
        labels, unknown_count = set(), 0
        for option in options:
            if not isinstance(option, dict) or set(option) != {"teksti", "fakti"}:
                raise ValueError("questions.json: alternativa është e pavlefshme.")
            label, fact = option["teksti"], option["fakti"]
            if not isinstance(label, str) or not label.strip() or label in labels:
                raise ValueError("questions.json: tekstet e alternativave duhet të jenë unike dhe jo bosh.")
            labels.add(label)
            if fact is None:
                unknown_count += 1
            elif not isinstance(fact, str) or fact not in facts or fact in covered_facts:
                raise ValueError("questions.json: fakti duhet të njihet dhe të përdoret vetëm një herë.")
            else:
                covered_facts.add(fact)
        if unknown_count != 1 or options[-1]["fakti"] is not None:
            raise ValueError("questions.json: alternativa e fundit duhet të jetë e vetmja 'Nuk e di'.")
        question_ids.add(question["id"])
    if covered_facts != facts:
        raise ValueError("questions.json: çdo fakt i vëzhgueshëm duhet të ketë një alternativë.")
    return facts, parsed_rules, questionnaire


FACTS_TEXT = (DATA_DIR / "seminar4/facts.json").read_text(encoding="utf-8")
RULES_TEXT = (DATA_DIR / "seminar4/rules.json").read_text(encoding="utf-8")
QUESTIONS_TEXT = (DATA_DIR / "seminar4/questions.json").read_text(encoding="utf-8")
known_facts, rules, questions = load_knowledge(FACTS_TEXT, RULES_TEXT, QUESTIONS_TEXT)
print(f"Baza e njohurive: {len(known_facts)} fakte të vëzhgueshme, {len(rules)} rregulla, {len(questions)} pyetje.")


Baza e njohurive: 13 fakte të vëzhgueshme, 14 rregulla, 5 pyetje.


## 3. Nga përgjigjet te përfundimet

Zgjedhjet kthehen në fakte dhe futen në një instancë të re të `ProductionSystem`. Shembulli `R02 → R03 → R04` tregon zinxhirin: shenja elektrike → dyshim për baterinë/lidhjet → këshillë për kontroll.

Nëse mungon një përgjigje, kërkohet plotësimi i formularit. Nëse të pesë përgjigjet janë dhënë, por asnjë rregull problemi nuk plotësohet, rezultati është **Nuk u gjet një gjendje e njohur**. Ky rezultat nuk provon se makina është pa problem. Nëse përputhen disa probleme, shfaqen të gjitha.


In [4]:
def infer_facts(initial_facts):
    """Krijo një sistem të ri për çdo vlerësim, pa trashëguar përgjigje të mëparshme."""
    initial_facts = set(initial_facts)
    if not initial_facts <= known_facts:
        raise ValueError("U dha një fakt jashtë fjalorit të vëzhgimeve.")
    system = ProductionSystem()
    for rule in rules:
        system.add_rule(rule["kushtet"], rule["perfundimet"])
    for fact in initial_facts:
        system.add_fact(fact)
    system.infer()
    matched = [rule for rule in rules if rule["kushtet"] <= system.working_memory]
    return {
        "faktet_fillestare": initial_facts,
        "faktet": set(system.working_memory),
        "probleme": sorted({fact for rule in matched if rule["lloji"] == "problem" for fact in rule["perfundimet"]}),
        "keshilla": sorted({fact for rule in matched if rule["lloji"] == "keshille" for fact in rule["perfundimet"]}),
        "rregullat": matched,
    }


def diagnose_answers(answers):
    """Kthe zgjedhjet në fakte; mungesa e përgjigjes nuk është përgjigjja 'Nuk e di'."""
    question_ids = {question["id"] for question in questions}
    if not isinstance(answers, dict) or not set(answers) <= question_ids:
        raise ValueError("Përgjigjet duhet t'u përkasin pesë pyetjeve të njohura.")
    selected_facts, missing = set(), []
    for question in questions:
        selected = answers.get(question["id"])
        if selected is None:
            missing.append(question["pyetja"])
            continue
        if type(selected) is not int or not 0 <= selected < len(question["alternativat"]):
            raise ValueError(f"Përgjigje e pavlefshme për pyetjen: {question['pyetja']}")
        fact = question["alternativat"][selected]["fakti"]
        if fact is not None:
            selected_facts.add(fact)
    if missing:
        return {"gjendja": "e_paplotesuar", "mungojne": missing, "probleme": [], "keshilla": []}
    result = infer_facts(selected_facts)
    return result | {"gjendja": "perfundim" if result["probleme"] else "e_panjohur"}


## 4. Çfarë problemi ka makina juaj?

Përgjigjuni të pesë pyetjeve, pastaj shtypni **Vlerëso përgjigjet**. Zgjidhni **Nuk e di** kur nuk e njihni gjendjen. **Fillo nga e para** pastron zgjedhjet. Ndryshimi i një përgjigjeje pastron rezultatin e vjetër; shtypni përsëri butonin për vlerësim të ri.

Rezultati shfaq faktet e vëzhguara, problemet e mundshme, këshillat dhe rregullat e plotësuara me shpjegimet e tyre.


In [5]:
UNANSWERED = -1
answer_controls = {}
question_rows = []
for index, question in enumerate(questions, start=1):
    control = widgets.Dropdown(
        options=[("Zgjidhni një përgjigje…", UNANSWERED)]
        + [(option["teksti"], option_index) for option_index, option in enumerate(question["alternativat"])],
        value=UNANSWERED,
        layout=widgets.Layout(width="100%"),
    )
    answer_controls[question["id"]] = control
    question_rows.append(widgets.VBox([
        widgets.Label(value=f"{index}. {question['pyetja']}"),
        control,
    ]))

diagnose_button = widgets.Button(description="Vlerëso përgjigjet", button_style="primary")
reset_button = widgets.Button(description="Fillo nga e para")
result_output = widgets.Output()


def show_diagnosis(_=None):
    result = diagnose_answers({
        key: None if control.value == UNANSWERED else control.value
        for key, control in answer_controls.items()
    })
    with result_output:
        result_output.clear_output(wait=True)
        if result["gjendja"] == "e_paplotesuar":
            print("Përgjigjuni të pesë pyetjeve. Nëse nuk e dini përgjigjen, zgjidhni 'Nuk e di'.")
            return result
        print("Faktet e vëzhguara:")
        for fact in sorted(result["faktet_fillestare"]):
            print("•", fact)
        if result["gjendja"] == "e_panjohur":
            print("Nuk u gjet një gjendje e njohur nga këto përgjigje.")
            print("Kjo nuk provon se makina nuk ka problem; baza e rregullave është e kufizuar.")
        else:
            print("\nProblemet e mundshme:")
            for problem in result["probleme"]:
                print("•", problem)
            print("\nHapat e rekomanduar:")
            for advice in result["keshilla"]:
                print("•", advice)
            print("\nRregullat e plotësuara dhe arsyetimi:")
            display(pd.DataFrame([
                {"Rregulli": rule["id"], "Kushtet": "; ".join(sorted(rule["kushtet"])),
                 "Përfundimet": "; ".join(sorted(rule["perfundimet"])), "Arsyetimi": rule["shpjegimi"]}
                for rule in result["rregullat"]
            ]))
    return result


def clear_diagnosis(_=None):
    result_output.clear_output(wait=False)


def reset_questionnaire(_=None):
    for control in answer_controls.values():
        control.value = UNANSWERED
    clear_diagnosis()


for control in answer_controls.values():
    control.observe(clear_diagnosis, names="value")
diagnose_button.on_click(show_diagnosis)
reset_button.on_click(reset_questionnaire)
questionnaire = widgets.VBox([
    widgets.HTML("<h3>Çfarë problemi ka makina juaj?</h3><p>Zgjidhni një alternativë për secilën nga pesë pyetjet.</p>"),
    *question_rows,
    widgets.HBox([diagnose_button, reset_button]),
    result_output,
])
display(questionnaire)


## 5. Kontrollet e skenarëve

Kontrollet mbulojnë baterinë/lidhjet elektrike, qarkun e ndezjes, mungesën e mundshme të karburantit, mbinxehjen dhe gomat. Përfshihen përgjigjet normale, **Nuk e di**, formulari i paplotë, alternativat e pavlefshme, disa probleme njëkohësisht dhe rivlerësimi pa memorie të mbetur.


In [6]:
# Rastet e pranimit: pesë përgjigje -> fakte -> përfundime në shqip.
normal_answers = {"ndezja": 3, "dritat": 1, "karburanti": 1, "temperatura": 1, "gomat": 2}
battery_answers = normal_answers | {"ndezja": 1, "dritat": 0}
assert "Dyshohet bateri e dobët ose lidhje elektrike e lirshme" in diagnose_answers(battery_answers)["probleme"]
assert "Dyshohet problem në qarkun e ndezjes ose te motorinoja" in diagnose_answers(normal_answers | {"ndezja": 0})["probleme"]
assert "Dyshohet mungesë karburanti" in diagnose_answers(normal_answers | {"ndezja": 2, "karburanti": 0})["probleme"]
assert "Ka shenja mbinxehjeje të motorit" in diagnose_answers(normal_answers | {"temperatura": 0})["probleme"]
assert "Goma e shfryrë kërkon kontroll" in diagnose_answers(normal_answers | {"gomat": 0})["probleme"]
assert "Dyshohet rrjedhje ajri nga goma ose valvula" in diagnose_answers(normal_answers | {"gomat": 1})["probleme"]
assert diagnose_answers(normal_answers)["gjendja"] == "e_panjohur"
unknown_answers = {question["id"]: len(question["alternativat"]) - 1 for question in questions}
unknown_result = diagnose_answers(unknown_answers)
assert unknown_result["gjendja"] == "e_panjohur"
assert unknown_result["faktet_fillestare"] == set()
assert diagnose_answers({})["gjendja"] == "e_paplotesuar"
assert diagnose_answers(normal_answers | {"ndezja": None})["gjendja"] == "e_paplotesuar"
assert diagnose_answers(normal_answers | {"ndezja": 0, "dritat": 2})["gjendja"] == "e_panjohur"

# Çdo vlerësim krijon memorie të re; përgjigjet e mëparshme nuk trashëgohen.
assert diagnose_answers(battery_answers) == diagnose_answers(battery_answers)
assert not diagnose_answers(normal_answers)["probleme"]
combined = diagnose_answers(battery_answers | {"temperatura": 0, "gomat": 0})
assert set(["Dyshohet bateri e dobët ose lidhje elektrike e lirshme", "Ka shenja mbinxehjeje të motorit", "Goma e shfryrë kërkon kontroll"]) <= set(combined["probleme"])
assert {"R02", "R03", "R04"} <= {rule["id"] for rule in combined["rregullat"]}
for invalid in [
    normal_answers | {"ndezja": -1},
    normal_answers | {"ndezja": 100},
    normal_answers | {"ndezja": True},
    normal_answers | {"ndezja": "1"},
    normal_answers | {"tjeter": 0},
]:
    try:
        diagnose_answers(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("U pranua një përgjigje e pavlefshme.")

# Kufijtë e motorit të inferencës: pa fakte, fakte të përsëritura, rregulla ciklike.
empty_system = ProductionSystem()
empty_system.infer()
assert empty_system.working_memory == set()
cyclic_system = ProductionSystem()
cyclic_system.add_fact("Fakti A")
cyclic_system.add_fact("Fakti A")
cyclic_system.add_rule({"Fakti A"}, {"Fakti B"})
cyclic_system.add_rule({"Fakti B"}, {"Fakti A"})
cyclic_system.infer()
assert cyclic_system.working_memory == {"Fakti A", "Fakti B"}
cyclic_system.infer()
assert cyclic_system.working_memory == {"Fakti A", "Fakti B"}
assert find_data_dir(DATA_DIR.parent) == find_data_dir(DATA_DIR.parent / "notebooks") == DATA_DIR
print("Kaluan kontrollet e skenarëve, përgjigjeve të panjohura dhe kufijve të inferencës.")


Kaluan kontrollet e skenarëve, përgjigjeve të panjohura dhe kufijve të inferencës.


## 6. Kontrollet e të dhënave dhe formularit

Provojmë të gjitha 540 kombinimet e alternativave dhe kontrollojmë që çdo përfundim të mbështetet nga rregullat. Provojmë edhe JSON të pavlefshëm dhe callback-un e formularit. Në fund zgjedhjet pastrohen që formulari të jetë gati për përdorim.


In [ ]:
# Provo të gjitha kombinimet e alternativave, përfshirë 'Nuk e di'.
matched_rule_ids = set()
combination_count = 0
for selection in product(*(range(len(question["alternativat"])) for question in questions)):
    answers = {question["id"]: choice for question, choice in zip(questions, selection, strict=True)}
    result = diagnose_answers(answers)
    assert result["gjendja"] != "e_paplotesuar"
    assert result["faktet_fillestare"] <= result["faktet"]
    assert bool(result["probleme"]) == (result["gjendja"] == "perfundim")
    assert all(rule["kushtet"] <= result["faktet"] for rule in result["rregullat"])
    supported = set().union(*(rule["perfundimet"] for rule in result["rregullat"]))
    assert result["faktet"] == result["faktet_fillestare"] | supported
    assert not {"Dyshohet bateri e dobët ose lidhje elektrike e lirshme", "Dyshohet problem në qarkun e ndezjes ose te motorinoja"} <= set(result["probleme"])
    matched_rule_ids.update(rule["id"] for rule in result["rregullat"])
    combination_count += 1
assert matched_rule_ids == {rule["id"] for rule in rules}

# Skedarët e keqformuar nuk duhet të krijojnë një bazë njohurish të pjesshme.
for invalid_facts in ["{}", "[]", '[""]', '["Fakti A", "Fakti A"]', "["]:
    try:
        load_knowledge(invalid_facts, RULES_TEXT, QUESTIONS_TEXT)
    except ValueError:
        pass
    else:
        raise AssertionError("U pranua fjalor faktesh i pavlefshëm.")

invalid_rule_sets = [[], [{}], [None]]
bad_rules = json.loads(RULES_TEXT)
invalid_rule_sets.append(bad_rules + [bad_rules[0]])
bad_rules = json.loads(RULES_TEXT)
bad_rules[0]["kushtet"] = ["Fakt që nuk ekziston"]
invalid_rule_sets.append(bad_rules)
bad_rules = json.loads(RULES_TEXT)
bad_rules[0]["perfundimet"] = [next(iter(known_facts))]
invalid_rule_sets.append(bad_rules)
for records in invalid_rule_sets:
    try:
        load_knowledge(FACTS_TEXT, json.dumps(records), QUESTIONS_TEXT)
    except ValueError:
        pass
    else:
        raise AssertionError("U pranuan rregulla të pavlefshme.")

invalid_question_sets = [[], [None] * 5]
bad_questions = json.loads(QUESTIONS_TEXT)
bad_questions[1]["id"] = bad_questions[0]["id"]
invalid_question_sets.append(bad_questions)
bad_questions = json.loads(QUESTIONS_TEXT)
bad_questions[0]["alternativat"][0]["fakti"] = "Fakt që nuk ekziston"
invalid_question_sets.append(bad_questions)
bad_questions = json.loads(QUESTIONS_TEXT)
bad_questions[0]["alternativat"][-1]["fakti"] = bad_questions[0]["alternativat"][0]["fakti"]
invalid_question_sets.append(bad_questions)
for records in invalid_question_sets:
    try:
        load_knowledge(FACTS_TEXT, RULES_TEXT, json.dumps(records))
    except ValueError:
        pass
    else:
        raise AssertionError("U pranua pyetësor i pavlefshëm.")

# Ekzekuto drejtpërdrejt callback-un e butonit që gabimet të mos fshihen nga widgets.
with capture_output():
    reset_questionnaire()
    assert show_diagnosis()["gjendja"] == "e_paplotesuar"
    for answers in (battery_answers, unknown_answers, normal_answers):
        for question_id, choice in answers.items():
            answer_controls[question_id].value = choice
        assert show_diagnosis() == diagnose_answers(answers)
    reset_questionnaire()
assert all(control.value == UNANSWERED for control in answer_controls.values())
assert all(control.index == 0 and control.label == 'Zgjidhni një përgjigje…' for control in answer_controls.values())
assert len(answer_controls) == 5
print(f"Kaluan {combination_count} kombinime, validimi i JSON-ve dhe kontrollet e formularit.")


Kaluan 540 kombinime, validimi i JSON-ve dhe kontrollet e formularit.


## Diskutim

Ky sistem nuk supozon se çdo problem ka një shkak të vetëm. Për shembull, shenjat e baterisë mund të shfaqen bashkë me një gomë të shfryrë. Përgjigjja **Nuk e di** nuk përdoret si fakt negativ dhe mungesa e një rregulli të përshtatshëm nuk nënkupton se makina është në rregull.

Për eksperimente, ndrysho alternativat dhe rregullat te `../data/seminar4/`, pastaj rindiz kernel-in dhe ekzekuto të gjitha qelizat. Ruaj dallimin mes vëzhgimit, përfundimit dhe këshillës.

Burime për shembujt: [RAC: makina nuk ndizet](https://www.rac.co.uk/drive/advice/know-how/help-my-car-wont-start-what-do-i-do/), [AA: mbinxehja dhe sistemi i ftohjes](https://www.theaa.com/breakdown-cover/advice/how-to-check-your-cars-cooling-fan), [AA: kapaku i ftohjes vetëm kur motori është ftohur](https://www.theaa.com/breakdown-cover/advice/how-to-check-your-engine-coolant), [AA: gomat që humbin presion](https://www.theaa.com/about-us/newsroom/14000-tyre-breakdowns-last-festive-season).

[Seminari 3](seminar3.ipynb) · [Indeksi](../README.md)
